In [1]:
import pandas as pd

elo = pd.read_csv("EloRatings.csv")
clean_matches = pd.read_csv("matches_clean.csv", low_memory=False, dtype={"MatchTime": str})
matches = pd.read_csv("Matches.csv", low_memory=False, dtype={"MatchTime": str})

In [46]:
years = matches["MatchDate"].map(lambda x: int(x.split("-")[0]))
filt = matches["HTHome"].isna().groupby([matches["Division"],]).sum()
filt

Division
ARG    5330
AUT    2306
B1        6
BRA    4850
CHN    2544
D1        0
D2        0
DEN    2634
E0        0
E1        1
E2        0
E3        0
EC        0
F1        1
F2        4
FIN    2354
G1       27
I1        2
I2        4
IRL    2367
JAP    4089
MEX    4080
N1        0
NOR    3092
P1        0
POL    3567
ROM    3640
RUS    3008
SC0       0
SC1       0
SC2       0
SC3       0
SP1       0
SP2      22
SUI    2273
SWE    3110
T1       36
USA    5233
Name: HTHome, dtype: int64

In [3]:
napoli_matches = matches[(matches["HomeTeam"] == "Napoli") | (matches["AwayTeam"] == "Napoli")]
napoli_matches.columns

Index(['Division', 'MatchDate', 'Season', 'HomeTeam', 'AwayTeam', 'FTHome',
       'FTAway', 'FTResult', 'HomeElo', 'AwayElo', 'Form3Home', 'Form5Home',
       'Form3Away', 'Form5Away', 'HTHome', 'HTAway', 'HTResult', 'HomeShots',
       'AwayShots', 'HomeTarget', 'AwayTarget', 'HomeFouls', 'AwayFouls',
       'HomeCorners', 'AwayCorners', 'HomeYellow', 'AwayYellow', 'HomeRed',
       'AwayRed', 'OddHome', 'OddDraw', 'OddAway', 'Over25', 'Under25',
       'HandiSize', 'HandiHome', 'HandiAway', 'C_LTH', 'C_LTA', 'C_VHD',
       'C_VAD', 'C_HTB', 'C_PHB', 'EloDiff', 'HomeAvgGoalsFor5',
       'HomeAvgGoalsAgainst5', 'AwayAvgGoalsFor5', 'AwayAvgGoalsAgainst5',
       'has_stats', 'has_handicap', 'has_ou', 'has_cluster_data', 'has_form',
       'has_elo', 'has_rolling_stats'],
      dtype='str')

## Validazione output Fase 1 (matches_clean.csv)

Controlli rispetto alla spec `docs/superpowers/specs/2026-07-12-dataset-feature-engineering-design.md`.

In [4]:
matches = pd.read_csv("matches_clean.csv", low_memory=False, parse_dates=["MatchDate"])

print(f"Shape: {matches.shape[0]:,} righe x {matches.shape[1]} colonne")
print(f"Range date: {matches['MatchDate'].min().date()} -> {matches['MatchDate'].max().date()}")
print(f"Campionati (Division): {matches['Division'].nunique()}")

Shape: 227,525 righe x 55 colonne
Range date: 2000-07-28 -> 2025-06-01
Campionati (Division): 38


In [5]:
# 1. Colonne scartate assenti, colonne attese presenti
dropped_expected = ["MatchTime", "MaxHome", "MaxDraw", "MaxAway", "MaxOver25", "MaxUnder25"]
still_present = [c for c in dropped_expected if c in matches.columns]
print("Colonne che dovevano essere scartate ma sono ancora presenti:", still_present or "nessuna (OK)")

required_new = ["EloDiff", "HomeAvgGoalsFor5", "HomeAvgGoalsAgainst5", "AwayAvgGoalsFor5", "AwayAvgGoalsAgainst5",
                "Season", "has_stats", "has_handicap", "has_ou", "has_cluster_data", "has_form",
                "has_elo", "has_rolling_stats"]
missing_new = [c for c in required_new if c not in matches.columns]
print("Colonne attese ma mancanti:", missing_new or "nessuna (OK)")

Colonne che dovevano essere scartate ma sono ancora presenti: nessuna (OK)
Colonne attese ma mancanti: nessuna (OK)


In [6]:
# 2. Righe scartate: target e quote non devono avere missing (spec: righe senza queste vanno droppate)
target_cols = ["FTHome", "FTAway", "FTResult"]
odds_cols = ["OddHome", "OddDraw", "OddAway"]
print("Missing nei target:", matches[target_cols].isna().sum().to_dict())
print("Missing nelle quote:", matches[odds_cols].isna().sum().to_dict())

# Duplicati esatti
print("Righe duplicate:", matches.duplicated().sum())

Missing nei target: {'FTHome': 0, 'FTAway': 0, 'FTResult': 0}
Missing nelle quote: {'OddHome': 0, 'OddDraw': 0, 'OddAway': 0}


Righe duplicate: 0


In [7]:
# 3. Copertura Elo dopo il fill (spec: 38.6% mancante in origine in Matches.csv)
elo_missing_before = 0.386
elo_missing_after = matches["HomeElo"].isna().mean() + matches["AwayElo"].isna().mean()
print(f"Elo ancora mancante dopo il fill (Home+Away, %): {elo_missing_after*100:.2f}")
print(f"has_elo == False: {(~matches['has_elo']).sum()} righe ({(~matches['has_elo']).mean()*100:.2f}%)")

# has_elo deve corrispondere esattamente a HomeElo/AwayElo non-null (post fill, prima di fillna(0) in add_elo_diff)
# NB: add_elo_diff riempie HomeElo/AwayElo mancanti con 0, quindi qui verifichiamo solo la coerenza del flag
print("Righe con has_elo=True ma HomeElo o AwayElo == 0:",
      ((matches["has_elo"]) & ((matches["HomeElo"] == 0) | (matches["AwayElo"] == 0))).sum())

Elo ancora mancante dopo il fill (Home+Away, %): 0.00
has_elo == False: 85501 righe (37.58%)
Righe con has_elo=True ma HomeElo o AwayElo == 0: 0


In [8]:
# 4. EloDiff = HomeElo - AwayElo
elo_diff_ok = (matches["EloDiff"] - (matches["HomeElo"] - matches["AwayElo"])).abs().max()
print(f"Max scarto EloDiff vs HomeElo-AwayElo: {elo_diff_ok:.6f} (deve essere ~0)")

Max scarto EloDiff vs HomeElo-AwayElo: 0.000000 (deve essere ~0)


In [9]:
# 5. Rolling goal stats: verifica manuale anti-leakage su una squadra (Napoli)
team = "Napoli"
team_matches = matches[(matches["HomeTeam"] == team) | (matches["AwayTeam"] == team)].sort_values("MatchDate").reset_index(drop=True)

goals_for, goals_against = [], []
for _, row in team_matches.iterrows():
    if row["HomeTeam"] == team:
        goals_for.append(row["FTHome"]); goals_against.append(row["FTAway"])
    else:
        goals_for.append(row["FTAway"]); goals_against.append(row["FTHome"])

window = 5
expected_avg_for, expected_avg_against = [], []
for i in range(len(team_matches)):
    prior_for = goals_for[max(0, i - window):i]
    prior_against = goals_against[max(0, i - window):i]
    expected_avg_for.append(sum(prior_for) / len(prior_for) if prior_for else 0)
    expected_avg_against.append(sum(prior_against) / len(prior_against) if prior_against else 0)

actual_avg_for = team_matches.apply(
    lambda r: r["HomeAvgGoalsFor5"] if r["HomeTeam"] == team else r["AwayAvgGoalsFor5"], axis=1
)
actual_avg_against = team_matches.apply(
    lambda r: r["HomeAvgGoalsAgainst5"] if r["HomeTeam"] == team else r["AwayAvgGoalsAgainst5"], axis=1
)

import numpy as np
diff_for = (actual_avg_for.values - np.array(expected_avg_for))
diff_against = (actual_avg_against.values - np.array(expected_avg_against))
print(f"Max scarto AvgGoalsFor5 (calcolo manuale vs dataset): {np.abs(diff_for).max():.6f}")
print(f"Max scarto AvgGoalsAgainst5 (calcolo manuale vs dataset): {np.abs(diff_against).max():.6f}")
print(f"Prima partita di {team} nel dataset ha rolling = 0 (nessuno storico)? "
      f"{actual_avg_for.iloc[0] == 0 and actual_avg_against.iloc[0] == 0}")

Max scarto AvgGoalsFor5 (calcolo manuale vs dataset): 0.000000
Max scarto AvgGoalsAgainst5 (calcolo manuale vs dataset): 0.000000
Prima partita di Napoli nel dataset ha rolling = 0 (nessuno storico)? True


In [10]:
# 6. Season: confine a luglio, formato "YYYY-YYYY+1"
sample = matches[["MatchDate", "Season"]].sample(2000, random_state=0)
def expected_season(d):
    y = d.year
    return f"{y}-{y+1}" if d.month >= 7 else f"{y-1}-{y}"
bad_season = sample[sample.apply(lambda r: expected_season(r["MatchDate"]) != r["Season"], axis=1)]
print("Righe con Season non conforme (su 2000 campionate):", len(bad_season))

# 7. Ordinamento per MatchDate
print("Dataset ordinato per MatchDate:", matches["MatchDate"].is_monotonic_increasing)

Righe con Season non conforme (su 2000 campionate): 0
Dataset ordinato per MatchDate: True


In [11]:
# 8. has_rolling_stats: coerenza con la presenza di almeno `window` partite precedenti per entrambe le squadre
# (qui verifichiamo solo che, quando True, i valori rolling non siano frutto di riempimento a 0 per mancanza dati)
inconsistent = matches[
    matches["has_rolling_stats"]
    & (matches["HomeAvgGoalsFor5"].isna() | matches["AwayAvgGoalsFor5"].isna())
]
print("Righe has_rolling_stats=True con valori rolling NaN:", len(inconsistent))
print("Quota partite con has_rolling_stats=True:", matches["has_rolling_stats"].mean().round(4))

# 9. Flag di disponibilità coerenti con NaN nelle colonne sorgente originali (spot check su has_stats)
stats_cols = ["HomeShots", "AwayShots", "HomeTarget", "AwayTarget", "HomeFouls", "AwayFouls",
              "HomeCorners", "AwayCorners", "HomeYellow", "AwayYellow", "HomeRed", "AwayRed",
              "HTHome", "HTAway", "HTResult"]
# dopo il fill, i missing diventano 0/"Unknown", quindi non possiamo ricontrollare da qui;
# verifichiamo solo che non restino NaN residui nelle colonne con flag
print("NaN residui nelle colonne has_stats:", matches[stats_cols].isna().sum().sum())

Righe has_rolling_stats=True con valori rolling NaN: 0
Quota partite con has_rolling_stats=True: 0.9807
NaN residui nelle colonne has_stats: 0
